# Phase 1 — ASR Head Training v2 (cached mHuBERT features + CTC)

Emergency Call Intelligence System — PoC

**Design:** because the backbone is fully frozen, the mHuBERT-147 outputs are computed **once** and written to disk (Stage A). Training (Stage B) only fits the 768->30 linear CTC head over that cache, so 30 epochs take minutes on a GPU.

**Requirements:** about 24 GB VRAM (8 GB is enough for Stage A), about 40 GB RAM and **about 30 GB of free disk** (feature cache: train about 27 GB plus dev about 0.8 GB, float16).

**Usage:**
1. Set `MAX_TRAIN_SAMPLES` in the config cell (`512` for a quick smoke test, `None` for the real run).
2. Run All. Stage A skips itself when the cache exists, so restarting after an interruption is safe (a half-finished cache is deleted, a complete one is left alone).
3. Watch the epoch time and dev WER in the `[TRAIN]` lines. The best model is saved to `OUTPUT_DIR`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# 1. Dependencies, once per fresh runtime
!pip install -q "transformers>=4.44" "datasets>=2.20" jiwer torchaudio soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 104.9 MB/s eta 0:00:00


In [8]:
# 2. Import & config
import os, json, time, math, itertools, statistics
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

from datasets import load_dataset, Audio
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, HubertModel
import jiwer

# ================= CONFIG =================
OUTPUT_DIR        = "/content/drive/MyDrive/CLEAR/phase1_ctc"
BACKBONE_ID       = "utter-project/mHuBERT-147"
DATASET_ID        = "openslr/librispeech_asr"
SR                = 16_000
HID               = 768                  # mHuBERT-base hidden dim

CACHE_DIR         = "./feat_cache"       # ~30 GB (full run)

MAX_TRAIN_SAMPLES = 2048                 # 512 = smoke test, None = full train-clean-100
MAX_EVAL_SAMPLES  = None                 # None means all of dev-clean (2703 samples)

PRECOMPUTE_BATCH  = 64                  # Stage A forward batch (utterances)
TRAIN_BATCH       = 256                   # Stage B batch (over the cache, comfortable)
NUM_EPOCHS        = 100
LR                = 2e-3
LR_PATIENCE       = 10                    # ReduceLROnPlateau (dev-WER)
LR_FACTOR         = 0.5
STOP_PATIENCE     = 3                    # early stopping (dev-WER)
NUM_WORKERS       = os.cpu_count()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[ENV] torch={torch.__version__} device={DEVICE}"
      + (f" gpu={torch.cuda.get_device_name(0)}" if DEVICE == "cuda" else ""))

[ENV] torch=2.11.0+cu128 device=cuda gpu=NVIDIA L4


In [3]:
# 3. Vocab & tokenizer (30 tokens: 27 characters + | + [UNK] + [PAD]=blank)
chars = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ'")
vocab = {c: i for i, c in enumerate(chars)}
vocab["|"]     = len(vocab)   # word separator (space)
vocab["[UNK]"] = len(vocab)
vocab["[PAD]"] = len(vocab)   # CTC blank
with open("vocab.json", "w") as f:
    json.dump(vocab, f)

tokenizer = Wav2Vec2CTCTokenizer("vocab.json", unk_token="[UNK]",
                                 pad_token="[PAD]", word_delimiter_token="|")
feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1, sampling_rate=SR, padding_value=0.0,
    do_normalize=True, return_attention_mask=True)

VOCAB_SIZE = len(vocab)
BLANK_ID   = vocab["[PAD]"]
ID2CH      = {i: c for c, i in vocab.items()}
print(f"[VOCAB] {VOCAB_SIZE} tokens, blank={BLANK_ID}")

[VOCAB] 30 token, blank=29


In [4]:
# 4. STAGE A - compute the mHuBERT outputs once and write them to disk
# Format: <tag>.f16 holds consecutive [T_i x 768] float16 blocks, <tag>.json holds lengths, labels and text.
def precompute_split(backbone, split, tag, max_samples=None):
    feat_path = os.path.join(CACHE_DIR, f"{tag}.f16")
    meta_path = os.path.join(CACHE_DIR, f"{tag}.json")
    if os.path.exists(meta_path):
        print(f"[CACHE] {tag}: ready, skipping")
        return
    if os.path.exists(feat_path):
        os.remove(feat_path)  # a half-finished earlier attempt

    stream = load_dataset(DATASET_ID, "clean", split=split, streaming=True)
    stream = stream.cast_column("audio", Audio(sampling_rate=SR))
    it = itertools.islice(stream, max_samples) if max_samples else iter(stream)

    lengths, labels, texts = [], [], []
    t0, done, sec_audio = time.perf_counter(), 0, 0.0
    fout = open(feat_path, "ab")

    def flush(buf_audio, buf_text):
        nonlocal done, sec_audio
        enc = feature_extractor(buf_audio, sampling_rate=SR, padding=True,
                                return_tensors="pt", return_attention_mask=True)
        iv = enc.input_values.to(DEVICE, dtype=torch.float16)
        am = enc.attention_mask.to(DEVICE)
        with torch.no_grad():
            out = backbone(iv, attention_mask=am, output_hidden_states=True)
            hs = out.hidden_states[9]      # layer 5/7/9 (0=embedding, 12=last)
        valid = backbone._get_feat_extract_output_lengths(am.sum(-1)).tolist()
        hs = hs.cpu().numpy()
        for i, (T, txt) in enumerate(zip(valid, buf_text)):
            hs[i, :T].astype(np.float16).tofile(fout)
            lengths.append(int(T))
            labels.append(tokenizer(txt.upper()).input_ids)
            texts.append(txt.upper())
            done += 1
            sec_audio += len(buf_audio[i]) / SR
        if done % 240 < len(buf_audio):
            el = time.perf_counter() - t0
            print(f"[PRE] {tag}: {done} samples | {sec_audio/3600:.2f} hours of audio | "
                  f"{sec_audio/max(el,1e-9):.0f}x real time")

    buf_a, buf_t = [], []
    for ex in it:
        buf_a.append(ex["audio"]["array"].astype(np.float16))
        buf_t.append(ex["text"])
        if len(buf_a) == PRECOMPUTE_BATCH:
            flush(buf_a, buf_t); buf_a, buf_t = [], []
    if buf_a:
        flush(buf_a, buf_t)
    fout.close()

    with open(meta_path, "w") as f:
        json.dump({"lengths": lengths, "labels": labels, "texts": texts, "dim": HID}, f)
    gb = os.path.getsize(feat_path) / 1e9
    print(f"[PRE] {tag}: done - {len(lengths)} samples, {gb:.1f} GB, "
          f"{(time.perf_counter()-t0)/60:.1f} min")


os.makedirs(CACHE_DIR, exist_ok=True)
need = [t for t in ("train", "dev")
        if not os.path.exists(os.path.join(CACHE_DIR, f"{t}.json"))]
if need:
    print(f"[PRE] loading the backbone ({BACKBONE_ID}, fp16)...")
    backbone = HubertModel.from_pretrained(
        BACKBONE_ID, torch_dtype=torch.float16).to(DEVICE).eval()
    if "train" in need:
        precompute_split(backbone, "train.100", "train", MAX_TRAIN_SAMPLES)
    if "dev" in need:
        precompute_split(backbone, "validation", "dev", MAX_EVAL_SAMPLES)
    del backbone
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
else:
    print("[PRE] the whole cache is ready")

[PRE] backbone yükleniyor (utter-project/mHuBERT-147, fp16)...


config.json:   0%|          | 0.00/1.59k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

...

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

[PRE] train: 256 örnek | 0.95 saat ses | 189x gerçek zamanlı
[PRE] train: 512 örnek | 1.86 saat ses | 216x gerçek zamanlı
[PRE] train: 768 örnek | 2.69 saat ses | 224x gerçek zamanlı
[PRE] train: 960 örnek | 3.39 saat ses | 231x gerçek zamanlı
[PRE] train: 1216 örnek | 4.30 saat ses | 235x gerçek zamanlı
[PRE] train: 1472 örnek | 5.26 saat ses | 239x gerçek zamanlı
[PRE] train: 1728 örnek | 6.18 saat ses | 243x gerçek zamanlı
[PRE] train: 1920 örnek | 6.86 saat ses | 245x gerçek zamanlı
[PRE] train: bitti — 2048 örnek, 2.0 GB, 1.8 dk


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

[PRE] dev: 256 örnek | 0.44 saat ses | 73x gerçek zamanlı
[PRE] dev: 512 örnek | 0.91 saat ses | 94x gerçek zamanlı
[PRE] dev: 768 örnek | 1.46 saat ses | 109x gerçek zamanlı
[PRE] dev: 960 örnek | 1.77 saat ses | 114x gerçek zamanlı
[PRE] dev: 1216 örnek | 2.37 saat ses | 116x gerçek zamanlı
[PRE] dev: 1472 örnek | 2.96 saat ses | 117x gerçek zamanlı
[PRE] dev: 1728 örnek | 3.33 saat ses | 119x gerçek zamanlı
[PRE] dev: 1920 örnek | 3.63 saat ses | 120x gerçek zamanlı
[PRE] dev: 2176 örnek | 4.16 saat ses | 120x gerçek zamanlı
[PRE] dev: 2432 örnek | 4.74 saat ses | 124x gerçek zamanlı
[PRE] dev: 2688 örnek | 5.35 saat ses | 125x gerçek zamanlı
[PRE] dev: bitti — 2703 örnek, 1.5 GB, 2.6 dk


In [5]:
# 5. Dataset over the cache, length-bucketed batch sampler and collate
class FeatDataset(Dataset):
    def __init__(self, tag):
        with open(os.path.join(CACHE_DIR, f"{tag}.json")) as f:
            meta = json.load(f)
        self.lengths = meta["lengths"]
        self.labels  = meta["labels"]
        self.texts   = meta["texts"]
        self.offsets = np.concatenate([[0], np.cumsum(self.lengths)])[:-1]
        self.mm = np.memmap(os.path.join(CACHE_DIR, f"{tag}.f16"),
                            dtype=np.float16, mode="r").reshape(-1, HID)

    def __len__(self):
        return len(self.lengths)

    def __getitem__(self, i):
        o, L = int(self.offsets[i]), self.lengths[i]
        x = torch.from_numpy(np.array(self.mm[o:o+L]))
        return x, torch.tensor(self.labels[i], dtype=torch.long), L, i


class BucketBatchSampler(Sampler):
    """Per epoch: mild randomness plus length bucketing, so we get both shuffling and little padding."""
    def __init__(self, lengths, batch_size, shuffle=True):
        self.lengths = np.array(lengths)
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __iter__(self):
        if self.shuffle:
            # add noise to the length, then sort, so similar lengths stay close together
            # but not in exactly the same groups, they differ every epoch
            noise = np.random.rand(len(self.lengths))
            order = np.lexsort((noise, self.lengths))   # length first, random on ties
            batches = [order[i:i+self.batch_size].tolist()
                       for i in range(0, len(order), self.batch_size)]
            np.random.shuffle(batches)                  # shuffle the batch order as well
        else:
            order = np.argsort(self.lengths)
            batches = [order[i:i+self.batch_size].tolist()
                       for i in range(0, len(order), self.batch_size)]
        yield from batches

    def __len__(self):
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size


def collate(items):
    feats, labs, lens, idxs = zip(*items)
    B, Tmax = len(feats), max(lens)
    Smax = max(len(l) for l in labs)
    x = torch.zeros(B, Tmax, HID)
    y = torch.full((B, Smax), BLANK_ID, dtype=torch.long)
    for i, (f, l) in enumerate(zip(feats, labs)):
        x[i, :f.shape[0]] = f
        y[i, :len(l)] = l
    return (x, y, torch.tensor(lens, dtype=torch.long),
            torch.tensor([len(l) for l in labs], dtype=torch.long),
            list(idxs))


train_ds = FeatDataset("train")
dev_ds   = FeatDataset("dev")
train_dl = DataLoader(train_ds, batch_sampler=BucketBatchSampler(train_ds.lengths, TRAIN_BATCH),
                      collate_fn=collate, num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"))
dev_dl   = DataLoader(dev_ds, batch_sampler=BucketBatchSampler(dev_ds.lengths, TRAIN_BATCH, shuffle=False),
                      collate_fn=collate, num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"))
print(f"[DATA] train={len(train_ds)} dev={len(dev_ds)} | "
      f"{len(train_dl)} train batch/epoch")

[DATA] train=2048 dev=2703 | 8 train batch/epoch


In [6]:
# 6. Model (linear CTC head), greedy decode, WER
class CTCHead(nn.Module):
    def __init__(self, dim=HID, vocab_size=VOCAB_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ELU(),
            nn.Linear(dim, dim),
            nn.ELU(),
            nn.Linear(dim, vocab_size)
        )

    def forward(self, x):                 # [B,T,768] -> [B,T,V]
        return self.net(x)


def greedy_decode(ids):
    out, prev = [], -1
    for t in ids:
        if t != prev and t != BLANK_ID:
            out.append(ID2CH.get(t, ""))
        prev = t
    return "".join(out).replace("|", " ").strip()


@torch.no_grad()
def evaluate_wer(model, dl, ds):
    model.eval()
    hyps, refs = [], []
    for x, y, xlen, ylen, idxs in dl:
        logits = model(x.to(DEVICE))
        pred = logits.argmax(-1).cpu().numpy()
        for b, i in enumerate(idxs):
            hyps.append(greedy_decode(pred[b, :xlen[b]].tolist()))
            refs.append(ds.texts[i])
    return jiwer.wer(refs, hyps), jiwer.cer(refs, hyps), hyps, refs

In [9]:
# 7. STAGE B - training loop (with val measurement and overfit diagnosis)
model = CTCHead().to(DEVICE)
n_par = sum(p.numel() for p in model.parameters())
print(f"[MODEL] trainable params: {n_par:,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=LR_PATIENCE)
ctc = nn.CTCLoss(blank=BLANK_ID, reduction="mean", zero_infinity=True)

best_cer = float('inf')
os.makedirs(OUTPUT_DIR, exist_ok=True)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    t0, tot_loss, nb = time.perf_counter(), 0.0, 0
    for x, y, xlen, ylen, idxs in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        logp = logits.log_softmax(-1).transpose(0, 1)
        loss = ctc(logp, y, xlen.to(DEVICE), ylen.to(DEVICE))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        tot_loss += loss.item(); nb += 1

    pred = logits[0, :xlen[0]].argmax(-1).cpu().tolist()
    print(f"epoch {epoch:>3} | loss {tot_loss/nb:.3f} | {time.perf_counter()-t0:.1f}s")
    print(f"   HYP: {greedy_decode(pred)[:60]}")
    print(f"   REF: {train_ds.texts[idxs[0]][:60]}")

    # evaluate and save every epoch
    # val EVERY epoch (the scheduler and checkpointing depend on it)
    va_wer, va_cer, _, _ = evaluate_wer(model, dev_dl, dev_ds)
    lr_now = optimizer.param_groups[0]["lr"]

    # train only every 10 epochs (informational)
    if epoch % 10 == 0:
        tr_wer, tr_cer, _, _ = evaluate_wer(model, train_dl, train_ds)
        print(f"   >>> TRAIN cer {tr_cer*100:.1f}% | VAL cer {va_cer*100:.1f}% | lr {lr_now:.1e}")
    else:
        print(f"   >>> VAL cer {va_cer*100:.1f}% wer {va_wer*100:.1f}% | lr {lr_now:.1e}")

    scheduler.step(va_cer)
    if va_cer < best_cer:
        best_cer = va_cer
        tokenizer.save_pretrained(OUTPUT_DIR)
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "ctc_head_best.pt"))
        print(f"   [SAVE] new best val-CER {va_cer*100:.1f}% -> ctc_head_best.pt")

[MODEL] trainable params: 1,204,254
epoch   1 | loss 4.464 | 4.4s
   HYP: H[UNK][UNK][UNK][UNK][UNK][UNK][UNK][UNK]
   REF: AND THERE WERE TWO BRANCH TUNNELS LEADING UP TO THE SURFACE 
   >>> VAL cer 92.9% wer 100.0% | lr 2.0e-03
   [KAYIT] yeni en iyi val-CER 92.9% -> ctc_head_best.pt
epoch   2 | loss 3.189 | 4.4s
   HYP: [UNK][UNK][UNK] [UNK]     [UNK][UNK][UNK]   [UNK][UNK][UNK][
   REF: MISTER GATHERCOLE SAID THE GIRL QUICKLY FISHER NODDED YES MI
   >>> VAL cer 89.6% wer 100.0% | lr 2.0e-03
   [KAYIT] yeni en iyi val-CER 89.6% -> ctc_head_best.pt
epoch   3 | loss 2.909 | 3.6s
   HYP: [UNK][UNK]  [UNK] [UNK][UNK] [UNK][UNK] [UNK][UNK] [UNK]L
   REF: THY LOVING HEART IN THE NOON AND THE AFTERNOON OF LIFE WE ST
   >>> VAL cer 91.6% wer 100.0% | lr 2.0e-03
epoch   4 | loss 2.771 | 3.8s
   HYP: OL SOSOS   L SS N S N OLN OOOO
   REF: SHE WAS PERFECTLY KIND TO SUSIE IT WAS AS IF SHE POSITIVELY 
   >>> VAL cer 85.9% wer 99.9% | lr 2.0e-03
   [KAYIT] yeni en iyi val-CER 85.9% -> ctc_head_be

In [ ]:
# 8. (Optional) test-clean evaluation. Roadmap target is WER below 10%
RUN_TEST_EVAL = False   # set to True and run the cell (about 1.4 GB of extra cache)

if RUN_TEST_EVAL:
    if not os.path.exists(os.path.join(CACHE_DIR, "test.json")):
        backbone = HubertModel.from_pretrained(
            BACKBONE_ID, torch_dtype=torch.float16).to(DEVICE).eval()
        precompute_split(backbone, "test.clean", "test", None)
        del backbone
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    test_ds = FeatDataset("test")
    test_dl = DataLoader(test_ds,
                         batch_sampler=BucketBatchSampler(test_ds.lengths, TRAIN_BATCH, shuffle=False),
                         collate_fn=collate, num_workers=NUM_WORKERS,
                         pin_memory=(DEVICE=="cuda"))
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "ctc_head_best.pt"),
                                     map_location=DEVICE))
    test_wer, _, _ = evaluate_wer(model, test_dl, test_ds)
    print(f"[TEST] test-clean WER: {test_wer*100:.2f}%  (target < 10%)")

### Notes

- **Stage A is a one-off cost.** On an L4 with 24 GB the fp16 forward runs at roughly 100-200x real time, so 100 hours of audio takes about 40-60 minutes. If it is interrupted, Run All again is safe because a completed cache is skipped.
- **Stage B is very cheap.** For a 23K-parameter head an epoch takes seconds to tens of seconds, and all 30 epochs take minutes. The bottleneck is now the memmap disk read, and with 40 GB of RAM the OS page cache holds most of the train cache, so it speeds up from the second epoch on.
- **The inference chain:** audio -> feature_extractor -> mHuBERT (fp16, frozen) -> `ctc_head_best.pt` -> greedy decode. The forward inside `flush` in Stage A plus `greedy_decode` in cell 6 are exactly this chain.
- **Smoke test:** `MAX_TRAIN_SAMPLES=512` and `MAX_EVAL_SAMPLES=256` run end to end in about 5 minutes. Then delete the cache folder (`!rm -rf feat_cache`) and switch to the real run with `None`.
- A WER below 10% with a purely linear head is an aggressive target. If convergence stalls, adding one hidden layer to `CTCHead` (768->768 GELU->30) gives a clear gain for a small departure from the roadmap.